In [67]:
import pandas as pd

# ================== 工具函数 ==================
# 颜色渲染：正数红色，其余默认
def colorize(val: float, width=10):
    if pd.isna(val):
        return " " * width
    s = f"{val:.3f}".rjust(width)  # 固定宽度，保证对齐
    if val > 0:
        return f"\033[91m{s}\033[0m"  # 红色
    return s

# 表格打印（对齐 + 颜色）
def print_colored_table(df: pd.DataFrame, title: str):
    print(title)
    # 打印列名
    header = " " * 10 + "".join(c.rjust(10) for c in df.columns)
    print(header)
    # 打印每行
    for idx, row in df.iterrows():
        row_str = str(idx).ljust(10)
        for val in row:
            row_str += colorize(val, width=10)
        print(row_str)
    # print()


# ================== 主逻辑 ==================
df = pd.read_csv("./history_results/results8/result_summary.csv")
# df = pd.read_csv("./results/result_summary.csv")

df = df[df['Metric'].isin(['Recall', 'NDCG'])]

loss_list = [f"loss{i}" for i in range(1, 6)]
loss_list = [f"loss{i}" for i in [3,5]]
# topk_list = ["Top 3", "Top 5", "Top 10", "Top 20"]
topk_list = ["Top 10"]

# ========== 生成表 (平均提升百分比) ==========
def make_tables(df, label):
    for topk in topk_list:
        df_topk = df[df["TopK"] == topk]
        pivot = (
            df_topk[df_topk["Loss"].isin(loss_list)]
            .groupby(["Model", "Loss"])["RelDiff(%)"]
            .mean()
            .reset_index()
        )
        table = pivot.pivot(index="Loss", columns="Model", values="RelDiff(%)").round(3)
        print_colored_table(table, f"\n=== {label} | {topk} ===")


# print("========== 汇总表 ==========")

# 遍历 campus
for campus, df_c in df.groupby("Campus"):
    # print(f"\n########## Campus: {campus} ##########")
    make_tables(df_c, f"Results | Campus={campus}")



=== Results | Campus=campus_10 | Top 10 ===
                 NCL       SGL    SimGCL   XSimGCL
loss3          0.547     0.315     0.255     0.284
loss5          0.218    -0.122     0.635     0.063

=== Results | Campus=campus_102 | Top 10 ===
                 NCL       SGL    SimGCL   XSimGCL
loss3          0.471     0.502     1.122    -0.015
loss5          0.373     0.227     0.270    -0.450

=== Results | Campus=campus_143 | Top 10 ===
                 NCL       SGL    SimGCL   XSimGCL
loss3          0.589     0.608     0.825     0.167
loss5          1.032     0.077    -0.097     0.567

=== Results | Campus=campus_15 | Top 10 ===
                 NCL       SGL    SimGCL   XSimGCL
loss3         -0.435     0.397     0.809    -5.132
loss5         -0.219     0.007     1.074    -4.896

=== Results | Campus=campus_34 | Top 10 ===
                 NCL       SGL    SimGCL   XSimGCL
loss3          0.234    -1.307    -0.353    -0.293
loss5         -0.788     1.432     1.521     0.013


In [63]:
import pandas as pd

# ================== 工具函数 ==================
# 颜色渲染：正数红色，其余默认
def colorize(val: float, width=10):
    if pd.isna(val):
        return " " * width
    s = f"{val:.3f}".rjust(width)  # 固定宽度，保证对齐
    if val > 0:
        return f"\033[91m{s}\033[0m"  # 红色
    return s

# 表格打印（按列宽对齐 + 颜色）
def print_colored_table(df: pd.DataFrame, title: str, value_width: int = 10):
    print(title)
    # 动态确定行名列宽
    row_label_width = max(12, max((len(str(i)) for i in df.index), default=0) + 2)
    # 打印列名
    header = " " * row_label_width + "".join(str(c).rjust(value_width) for c in df.columns)
    print(header)
    # 打印每行
    for idx, row in df.iterrows():
        row_str = str(idx).ljust(row_label_width)
        for val in row:
            row_str += colorize(val, width=value_width)
        print(row_str)
    # print()


# ================== 主逻辑 ==================
# df = pd.read_csv("./history_results/results8/result_summary.csv")
# # df = pd.read_csv("./results/result_summary.csv")

# 确保数值列为数值类型
df["RelDiff(%)"] = pd.to_numeric(df["RelDiff(%)"], errors="coerce")

# 只考虑 loss3 和 loss5
loss_keep = ["loss3", "loss5"]
loss_keep = ["loss3"]
# topk_list = ["Top 3", "Top 5", "Top 10", "Top 20"]
topk_list = ["Top 10"]

# print("========== 汇总表 ==========")

# 过滤数据
subset = df[df["Loss"].isin(loss_keep)]

# 遍历 (Campus, Loss, TopK)，分别打印表格
for campus, df_c in subset.groupby("Campus", sort=True):
    # print(f"\n########## Campus: {campus} ##########")
    for loss in loss_keep:
        # print(f"###### Loss: {loss} ######")
        df_l = df_c[df_c["Loss"] == loss]
        for topk in topk_list:
            g = df_l[df_l["TopK"] == topk]
            if g.empty:
                continue
            pivot = (
                g.groupby(["Metric", "Model"])["RelDiff(%)"]
                 .mean()
                 .reset_index()
                 .pivot(index="Metric", columns="Model", values="RelDiff(%)")
                 .sort_index()
                 .round(3)
            )
            pivot = pivot.reindex(sorted(pivot.columns), axis=1)  # 模型列排序
            print_colored_table(
                pivot,
                f"=== Results | Campus={campus} | Loss={loss} | TopK={topk} ==="
            )


=== Results | Campus=campus_10 | Loss=loss3 | TopK=Top 10 ===
                   NCL       SGL    SimGCL   XSimGCL
NDCG             0.778     0.265     0.372     0.493
Recall           0.316     0.365     0.139     0.074
=== Results | Campus=campus_102 | Loss=loss3 | TopK=Top 10 ===
                   NCL       SGL    SimGCL   XSimGCL
NDCG             0.256     0.362     0.579     0.193
Recall           0.686     0.642     1.665    -0.222
=== Results | Campus=campus_143 | Loss=loss3 | TopK=Top 10 ===
                   NCL       SGL    SimGCL   XSimGCL
NDCG             0.436     0.214     0.760     0.042
Recall           0.742     1.003     0.890     0.291
=== Results | Campus=campus_15 | Loss=loss3 | TopK=Top 10 ===
                   NCL       SGL    SimGCL   XSimGCL
NDCG            -0.693     0.287     0.593    -3.204
Recall          -0.177     0.508     1.024    -7.061
=== Results | Campus=campus_34 | Loss=loss3 | TopK=Top 10 ===
                   NCL       SGL    SimGCL   XSimGCL

In [61]:
import pandas as pd
from colorama import Fore, Style

for model, subdf in df.groupby("Model"):
    print(f"\n===== Model: {model} =====")

    # 先转成字符串表格
    table_str = subdf.to_string(index=False)

    # 按行拆分
    lines = table_str.split("\n")

    # 第一行是表头，原样打印
    print(lines[0])

    # 从第二行开始逐行处理
    for i, (_, row) in enumerate(subdf.iterrows(), start=1):
        line = lines[i]
        if row["RelDiff(%)"] > 0:
            print(Fore.RED + line + Style.RESET_ALL)
        else:
            print(line)



===== Model: NCL =====
Model     Campus  Loss   TopK Metric  Baseline(loss0)   Value  AbsDiff  RelDiff(%)
  NCL  campus_10 loss1  Top 3 Recall          0.29537 0.28423 -0.01114   -3.771541
  NCL  campus_10 loss1  Top 3   NDCG          0.35446 0.33997 -0.01449   -4.087908
  NCL  campus_10 loss1  Top 5 Recall          0.38103 0.37129 -0.00974   -2.556229
  NCL  campus_10 loss1  Top 5   NDCG          0.37450 0.36115 -0.01335   -3.564753
  NCL  campus_10 loss1 Top 10 Recall          0.49609 0.49140 -0.00469   -0.945393
  NCL  campus_10 loss1 Top 10   NDCG          0.40876 0.39870 -0.01006   -2.461102
  NCL  campus_10 loss1 Top 20 Recall          0.60226 0.59346 -0.00880   -1.461163
  NCL  campus_10 loss1 Top 20   NDCG          0.44625 0.43514 -0.01111   -2.489636
  NCL  campus_10 loss2  Top 3 Recall          0.29537 0.29326 -0.00211   -0.714358
  NCL  campus_10 loss2  Top 3   NDCG          0.35446 0.35303 -0.00143   -0.403431
  NCL  campus_10 loss2  Top 5 Recall          0.38103 0.37630 -